# Deep dive 5/13: `marketing_campaigns`

Every column gets both a Bronze (raw) and Silver (typed) look -- not just "does it exist," but "does the cast actually do the right thing," using `typeof()` side-by-side the way you've been doing manually in the earlier notebooks.

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `marketing_campaigns_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 200 rows, Source: Internal, Partition: full_snapshot.

`contracts.py` currently gates nothing on this table beyond `campaign_id`/`campaign_status` presence -- no domains, no `campaign_name` in `expected_columns` even though the dictionary marks it NOT NULL. Section 5 checks the three domain candidates (`campaign_type`, `campaign_objective`, `campaign_status`) directly.

Run top to bottom. Checklist near the end; free cells after that.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. The whole table

200 rows -- small enough to look at directly, Bronze and Silver side by side.

In [ ]:
print("Bronze:")
display(q_full("SELECT * FROM bronze.marketing_campaigns ORDER BY campaign_id"))

In [ ]:
print("Silver:")
display(q_full("SELECT * FROM silver.dim_marketing_campaigns ORDER BY campaign_id"))

## 1. Column inventory -- names AND types, Bronze vs. Silver

In [ ]:
expected_silver_columns = [
    "campaign_id", "campaign_name", "description", "campaign_type", "campaign_objective",
    "promoted_product", "target_segment", "target_country", "start_date", "end_date", "budget",
    "campaign_status", "expected_conversion_rate",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_marketing_campaigns'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_marketing_campaigns:", missing or "none")
print("Extra in silver.dim_marketing_campaigns:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

In [ ]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='marketing_campaigns' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_marketing_campaigns' ORDER BY ordinal_position
"""))

## 2. Row-count reconciliation

In [ ]:
bronze_total = q("SELECT count(*) AS n FROM bronze.marketing_campaigns")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT campaign_id) AS n FROM bronze.marketing_campaigns")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_marketing_campaigns")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT campaign_id) AS n FROM silver.dim_marketing_campaigns")["n"][0]

print(f"bronze.marketing_campaigns:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct campaign_id")
print(f"silver.dim_marketing_campaigns:  {silver_total:,} rows, {silver_distinct_pk:,} distinct campaign_id")
print()
print("Dictionary declares 200 rows -- matches" if bronze_total == 200 else f"Dictionary declares 200 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

## 3. NOT NULL audit -- Bronze raw vs. Silver typed, both directions

Dictionary NOT NULL: `campaign_id`, `campaign_name`, `campaign_type`, `campaign_objective`, `start_date`, `end_date`, `campaign_status`. Nullable: `description`, `promoted_product`, `target_segment`, `target_country`, `budget`, `expected_conversion_rate`. Checking both Bronze (raw text blank/null) and Silver (post-cast null -- a cast failure would show up here even if Bronze looked populated).

In [ ]:
declared_not_null = [
    "campaign_id", "campaign_name", "campaign_type", "campaign_objective",
    "start_date", "end_date", "campaign_status",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.marketing_campaigns")["n"][0]
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.marketing_campaigns
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.dim_marketing_campaigns WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL",
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

## 4. Nullable columns -- populated rate, Bronze vs. Silver

Not a pass/fail check (these are legitimately nullable), but worth knowing the actual population rate and whether the cast changed it for the numeric/date ones.

In [ ]:
nullable_cols = ["description", "promoted_product", "target_segment", "target_country", "budget", "expected_conversion_rate"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.marketing_campaigns")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.marketing_campaigns
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.dim_marketing_campaigns WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total, "bronze_populated": bronze_populated,
                 "silver_populated": silver_populated,
                 "cast_dropped_values": bronze_populated - silver_populated})

pd.DataFrame(rows)

## 5. Domain checks -- none currently gated in `contracts.py`

In [ ]:
# campaign_type -- dictionary: Email, SMS, Push, WhatsApp, Voice, Mix
print("Bronze:")
display(q("SELECT campaign_type, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT campaign_type, count(*) AS n FROM silver.dim_marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# campaign_objective -- dictionary: Acquisition, Retention, Cross-sell, Up-sell, Reactivation
print("Bronze:")
display(q("SELECT campaign_objective, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT campaign_objective, count(*) AS n FROM silver.dim_marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# campaign_status -- dictionary: Planned, Active, Paused, Completed
print("Bronze:")
display(q("SELECT campaign_status, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))
print("Silver:")
display(q("SELECT campaign_status, count(*) AS n FROM silver.dim_marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

## 6. `target_country` canonicalization -- nullable, but still runs through `as_country()`

In [ ]:
print("Bronze raw:")
display(q("SELECT target_country, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

print("Silver (post-canonicalization):")
display(q("SELECT target_country, count(*) AS n FROM silver.dim_marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}
raw_values = q("SELECT DISTINCT lower(trim(target_country)) AS v FROM bronze.marketing_campaigns WHERE target_country IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw values NOT covered by COUNTRY_CANONICAL:", uncovered or "none -- full coverage")

## 7. `start_date`/`end_date` -- cast correctness and internal consistency, Bronze vs. Silver

In [ ]:
q("""
    SELECT
        typeof(start_date), typeof(end_date),
        count(*) FILTER (WHERE start_date IS NULL) AS start_nulls,
        count(*) FILTER (WHERE end_date IS NULL) AS end_nulls,
        count(*) FILTER (WHERE end_date <= start_date) AS ends_before_or_when_starts,
        min(start_date) AS earliest_start, max(start_date) AS latest_start,
        min(end_date) AS earliest_end, max(end_date) AS latest_end
    FROM silver.dim_marketing_campaigns
    GROUP BY ALL
""")

In [ ]:
# Same window check as every other table so far.
q("""
    SELECT
        count(*) FILTER (WHERE start_date < DATE '2023-06-17') AS start_before_window,
        count(*) FILTER (WHERE start_date > DATE '2026-06-17') AS start_after_window,
        count(*) FILTER (WHERE end_date > DATE '2026-06-17') AS end_after_window
    FROM silver.dim_marketing_campaigns
""")

## 8. `budget` -- range sanity, Bronze vs. Silver typing

In [ ]:
q("""
    SELECT
        typeof(budget) AS silver_type,
        min(budget) AS min_budget, max(budget) AS max_budget,
        count(*) FILTER (WHERE budget < 0) AS negative_budgets,
        count(*) FILTER (WHERE budget IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_marketing_campaigns
    GROUP BY ALL
""")

## 9. `expected_conversion_rate` -- range sanity + is it a percentage (0-100) or a fraction (0-1)?

In [ ]:
q("""
    SELECT
        typeof(expected_conversion_rate) AS silver_type,
        min(expected_conversion_rate) AS min_rate, max(expected_conversion_rate) AS max_rate,
        count(*) FILTER (WHERE expected_conversion_rate < 0) AS negative_rates,
        count(*) FILTER (WHERE expected_conversion_rate > 100) AS above_100,
        count(*) FILTER (WHERE expected_conversion_rate IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_marketing_campaigns
    GROUP BY ALL
""")

## 10. `promoted_product`/`target_segment` -- cross-check against real domains from other tables

Neither has a dictionary-declared value set of its own, but `target_segment` should plausibly overlap with `customers.segment` (`Premium, Plus, Basic, Student`) and `promoted_product` with `products.product_type` if these campaigns are actually targeting real product/segment categories.

In [ ]:
print("promoted_product (Bronze):")
display(q("SELECT promoted_product, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

print("target_segment (Bronze):")
display(q("SELECT target_segment, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 ORDER BY n DESC"))

In [ ]:
# Cross-check against the real product_type/segment domains found in earlier deep dives.
real_product_types = {"Cuenta Ahorro", "Tarjeta Crédito", "Cuenta Corriente", "Tarjeta Débito",
                       "Préstamo Personal", "Préstamo Hipotecario", "Inversión", "Seguro"}
real_segments = {"Premium", "Plus", "Basic", "Student"}

promoted = set(q("SELECT DISTINCT promoted_product AS v FROM bronze.marketing_campaigns WHERE promoted_product IS NOT NULL")["v"])
segments = set(q("SELECT DISTINCT target_segment AS v FROM bronze.marketing_campaigns WHERE target_segment IS NOT NULL")["v"])

print("promoted_product values NOT in the real products.product_type domain:", promoted - real_product_types or "none -- full overlap")
print("target_segment values NOT in the real customers.segment domain:", segments - real_segments or "none -- full overlap")

## 11. `description`/`campaign_name` -- free text spot-check + an unauthorized-but-worth-checking UNIQUE look at `campaign_name`

The dictionary doesn't declare `campaign_name` UNIQUE (only `campaign_id` is), but checking anyway costs nothing and two campaigns sharing an identical name would be worth knowing about regardless of whether it's a formal violation.

In [ ]:
q_full("SELECT campaign_name, count(*) AS n FROM bronze.marketing_campaigns GROUP BY 1 HAVING count(*) > 1 ORDER BY n DESC")

In [ ]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE description IS NULL OR trim(description) = '') AS description_nulls,
        avg(len(description)) AS avg_description_len,
        min(len(description)) AS min_description_len,
        max(len(description)) AS max_description_len
    FROM bronze.marketing_campaigns
""")

## Checklist

- [ ] Column inventory (1): spec/table agree, Bronze and Silver types both correct
- [ ] Row reconciliation (2): 200 rows as the dictionary declares
- [ ] NOT NULL audit (3): all 7 declared columns populated in both Bronze and Silver, no cast silently introduced a NULL
- [ ] Nullable columns (4): population rates make sense, no cast silently dropped a populated value
- [ ] Domain checks (5): `campaign_type`/`campaign_objective`/`campaign_status` match declared sets in both Bronze and Silver
- [ ] `target_country` (6): full canonicalization coverage
- [ ] Date fields (7): no campaign ends before/when it starts, note any window mismatch
- [ ] `budget` (8): no negatives
- [ ] `expected_conversion_rate` (9): confirm whether it's a 0-100 percentage or 0-1 fraction, no out-of-range values
- [ ] `promoted_product`/`target_segment` (10): note any value that doesn't overlap the real domains from `products`/`customers`
- [ ] `campaign_name`/`description` (11): note any duplicate names, sanity-check description length

In [ ]:
con.close()
print("closed")